# Project 2

## Configuration

In [10]:
import time
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")

Spark 4.1.0   catalog: lakehouse


DataFrame[]

In [11]:
BOOTSTRAP = "kafka:9092"
TOPIC     = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

In [12]:
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


## Bronze layer

In [13]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.bronze_trips (
    key STRING,
    value STRING,
    topic STRING,
    partition INT,
    offset BIGINT,
    timestamp TIMESTAMP
)
USING iceberg
""")

DataFrame[]

In [ ]:
# import shutil

# # Clean up any leftover files from a previous run of this cell
# shutil.rmtree("/tmp/bronze",     ignore_errors=True)
# shutil.rmtree("/tmp/chk-bronze", ignore_errors=True)

bronze_source = (
    raw_stream
    .select(
        F.col("key").cast("string"),
        F.col("value").cast("string"),
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.col("timestamp")
    )
)

bronze_query = (
    bronze_source.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/chk-bronze")
    .trigger(processingTime="5 seconds")
    .toTable("lakehouse.taxi.bronze_trips")
)

time.sleep(12)

count_before = spark.read.table("lakehouse.taxi.bronze_trips").count()
print(f"Row count before restart: {count_before}")

bronze_query.stop()
print("Query stopped.")

# Restart the query using the SAME checkpoint directory.
# Even though startingOffsets is still 'earliest', Spark ignores that setting
# and resumes from the committed offsets in the checkpoint.

bronze_query2 = (
    bronze_source.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/chk-bronze")
    .trigger(processingTime="5 seconds")
    .toTable("lakehouse.taxi.bronze_trips")
)

time.sleep(12)   # wait for two triggers

count_after = spark.read.table("lakehouse.taxi.bronze_trips").count()
print(f"Row count before restart : {count_before}")
print(f"Row count after restart  : {count_after}")
print()
if count_after == count_before:
    print("Counts are equal — the checkpoint prevented re-ingestion of already-processed messages.")
else:
    print(f"Counts differ by {count_after - count_before} — "
          "those are new messages produced between the two runs.")

bronze_query2.stop()

Row count before restart: 6008
Query stopped.
Row count before restart : 6008
Row count after restart  : 6033

Counts differ by 25 — those are new messages produced between the two runs.


In [59]:
spark.sql("""SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT topic, partition, offset) AS unique_events
FROM lakehouse.taxi.bronze_trips;""").show()

+----------+-------------+
|total_rows|unique_events|
+----------+-------------+
|      6033|         6033|
+----------+-------------+

